# CUAD Embedder Fine-Tune — bge-small-en-v1.5 (C2 retrieval)

Fine-tunes `BAAI/bge-small-en-v1.5` on (focused query → covering paragraph window)
pairs built by `docintel.scripts.build_embed_pairs` (eval contracts held out), then
exports an ONNX bundle for fastembed CPU serving.

**Inputs (upload to Drive):** `train.jsonl`, `dev.jsonl` from
`data/processed/embed_pairs/`. **Output:** `rag-embed-cuad.zip` →
laptop `models/rag-embed-cuad/`. Runtime: GPU (T4 is fine), ~20–40 min.

In [ ]:
%pip -q install -U sentence-transformers datasets "optimum[onnxruntime]" onnx

from google.colab import drive

drive.mount("/content/drive")
PAIRS_DIR = "/content/drive/MyDrive/docintel/embed_pairs"  # adjust to where you uploaded
OUT_DIR = "/content/drive/MyDrive/docintel"

In [ ]:
import json
from pathlib import Path


def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines()]


train_pairs = read_jsonl(f"{PAIRS_DIR}/train.jsonl")
dev_pairs = read_jsonl(f"{PAIRS_DIR}/dev.jsonl")
print(len(train_pairs), "train pairs,", len(dev_pairs), "dev pairs")

In [ ]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator

corpus, corpus_ids = {}, {}
for pair in dev_pairs:
    corpus_ids.setdefault(pair["positive"], f"d{len(corpus_ids)}")
    corpus[corpus_ids[pair["positive"]]] = pair["positive"]

queries, relevant = {}, {}
for i, pair in enumerate(dev_pairs):
    qid = f"q{i}"
    queries[qid] = pair["query"]
    relevant.setdefault(qid, set()).add(corpus_ids[pair["positive"]])

dev_evaluator = InformationRetrievalEvaluator(
    queries, corpus, relevant, name="cuad-dev", accuracy_at_k=[1, 3, 5], map_at_k=[5]
)

In [ ]:
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
print("baseline dev:", dev_evaluator(model))

train_ds = Dataset.from_list(
    [{"anchor": p["query"], "positive": p["positive"]} for p in train_pairs]
)
args = SentenceTransformerTrainingArguments(
    output_dir="/content/ckpt",
    num_train_epochs=2,
    per_device_train_batch_size=64,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MNRL: avoid duplicate positives in-batch
    logging_steps=50,
    eval_strategy="no",
    save_strategy="no",
    report_to=[],
)
trainer = SentenceTransformerTrainer(
    model=model, args=args, train_dataset=train_ds, loss=MultipleNegativesRankingLoss(model)
)
trainer.train()
print("fine-tuned dev:", dev_evaluator(model))

In [ ]:
ST_DIR, BUNDLE = "/content/bge-small-cuad", "/content/rag-embed-cuad"
model.save_pretrained(ST_DIR)

from optimum.onnxruntime import ORTModelForFeatureExtraction

ort_model = ORTModelForFeatureExtraction.from_pretrained(ST_DIR, export=True)
ort_model.save_pretrained(BUNDLE)  # writes model.onnx + config.json

import shutil

for name in ("tokenizer.json", "tokenizer_config.json", "special_tokens_map.json"):
    shutil.copy(f"{ST_DIR}/{name}", f"{BUNDLE}/{name}")

In [ ]:
import json

PARITY_SENTENCES = [
    "Governing Law: which state or country's law governs the agreement",
    "Non-Compete: restrictions on competing with the counterparty",
    "Insurance: requirement for one party to maintain insurance coverage",
    "This Agreement shall be governed by the laws of the State of New York.",
    "Licensee shall not solicit employees of Licensor during the term.",
    "Either party may terminate this Agreement upon thirty (30) days notice.",
    "All disputes shall be resolved by binding arbitration in London.",
    "The term of this Agreement is five (5) years from the Effective Date.",
]
vectors = model.encode(PARITY_SENTENCES, normalize_embeddings=True).tolist()
with open(f"{BUNDLE}/parity.json", "w", encoding="utf-8") as handle:
    json.dump({"sentences": PARITY_SENTENCES, "vectors": vectors}, handle)

In [ ]:
# In-Colab parity: ONNX (CLS pooling + L2 norm) vs the sentence-transformers model.
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BUNDLE)
encoded = tokenizer(PARITY_SENTENCES, padding=True, truncation=True, return_tensors="pt")
onnx_out = ort_model(**encoded).last_hidden_state[:, 0].numpy()  # CLS
onnx_out = onnx_out / np.linalg.norm(onnx_out, axis=1, keepdims=True)
st_out = np.asarray(vectors)
cosines = (onnx_out * st_out).sum(axis=1)
print("in-Colab parity cosines:", cosines.round(6))
assert cosines.min() >= 0.999, "ONNX export drifted from the trained model"

shutil.make_archive(f"{OUT_DIR}/rag-embed-cuad", "zip", BUNDLE)
print("bundle saved to Drive:", f"{OUT_DIR}/rag-embed-cuad.zip")